In [1]:
import itertools
import pandas as pd

df = pd.DataFrame({
    "avg_order_value":    [142.5, 89.0, 210.3, 67.8, 185.0],
    "discount_rate":      [0.10,  0.25, 0.05,  0.30, 0.15],
    "days_since_signup":  [120,   45,   380,   12,   200],
    "items_per_order":    [3.2,   1.8,  5.1,   1.2,  4.0],
    "return_rate":        [0.05,  0.18, 0.02,  0.22, 0.08],
})

numeric_cols = df.columns.tolist()

for col_a, col_b in itertools.combinations(numeric_cols, 2):
    feature_name = f"{col_a}_x_{col_b}"
    df[feature_name] = df[col_a] * df[col_b]

interaction_cols = [c for c in df.columns if "_x_" in c]
print(df[interaction_cols].head())


   avg_order_value_x_discount_rate  avg_order_value_x_days_since_signup  \
0                           14.250                              17100.0   
1                           22.250                               4005.0   
2                           10.515                              79914.0   
3                           20.340                                813.6   
4                           27.750                              37000.0   

   avg_order_value_x_items_per_order  avg_order_value_x_return_rate  \
0                             456.00                          7.125   
1                             160.20                         16.020   
2                            1072.53                          4.206   
3                              81.36                         14.916   
4                             740.00                         14.800   

   discount_rate_x_days_since_signup  discount_rate_x_items_per_order  \
0                              12.00             

In [2]:
import itertools

customer_segments = ["new", "returning", "vip"]
product_categories = ["electronics", "apparel", "home_goods", "beauty"]
channels = ["mobile", "desktop"]

# All segment × category × channel combinations
combos = list(itertools.product(customer_segments, product_categories, channels))

grid_df = pd.DataFrame(combos, columns=["segment", "category", "channel"])

# Simulate a conversion rate lookup per combination
import numpy as np
np.random.seed(7)
grid_df["avg_conversion_rate"] = np.round(
    np.random.uniform(0.02, 0.18, size=len(grid_df)), 3
)

print(grid_df.head(12))
print(f"\nTotal combinations: {len(grid_df)}")


      segment     category  channel  avg_conversion_rate
0         new  electronics   mobile                0.032
1         new  electronics  desktop                0.145
2         new      apparel   mobile                0.090
3         new      apparel  desktop                0.136
4         new   home_goods   mobile                0.176
5         new   home_goods  desktop                0.106
6         new       beauty   mobile                0.100
7         new       beauty  desktop                0.032
8   returning  electronics   mobile                0.063
9   returning  electronics  desktop                0.100
10  returning      apparel   mobile                0.129
11  returning      apparel  desktop                0.149

Total combinations: 24


In [3]:
import itertools

customer_features = [
    "customer_age", "days_since_signup", "lifetime_value",
    "total_orders", "avg_order_value"
]

product_features = [
    "category", "brand_tier", "avg_rating",
    "review_count", "is_sponsored"
]

behavioral_features = [
    "pages_viewed_last_7d", "search_queries_last_7d",
    "cart_abandonment_rate", "wishlist_size"
]

# Flatten all feature groups into one list
all_features = list(itertools.chain(
    customer_features,
    product_features,
    behavioral_features
))

print(f"Total features: {len(all_features)}")
print(all_features)


Total features: 14
['customer_age', 'days_since_signup', 'lifetime_value', 'total_orders', 'avg_order_value', 'category', 'brand_tier', 'avg_rating', 'review_count', 'is_sponsored', 'pages_viewed_last_7d', 'search_queries_last_7d', 'cart_abandonment_rate', 'wishlist_size']


In [4]:
import itertools

# Transaction history for customer C-10482, ordered chronologically
transactions = [
    {"order_id": "ORD-8821", "amount": 134.50, "items": 3},
    {"order_id": "ORD-8934", "amount":  89.00, "items": 2},
    {"order_id": "ORD-9102", "amount": 210.75, "items": 5},
    {"order_id": "ORD-9341", "amount":  55.20, "items": 1},
    {"order_id": "ORD-9488", "amount": 178.90, "items": 4},
    {"order_id": "ORD-9601", "amount": 302.10, "items": 7},
]

# Build lag-3 features for each transaction (using 3 most recent prior orders)
window_size = 3
features = []

for i in range(window_size, len(transactions)):
    window = list(itertools.islice(transactions, i - window_size, i))
    current = transactions[i]

    lag_amounts = [t["amount"] for t in window]
    features.append({
        "order_id":         current["order_id"],
        "current_amount":   current["amount"],
        "lag_1_amount":     lag_amounts[-1],
        "lag_2_amount":     lag_amounts[-2],
        "lag_3_amount":     lag_amounts[-3],
        "rolling_mean_3":   round(sum(lag_amounts) / len(lag_amounts), 2),
        "rolling_max_3":    max(lag_amounts),
    })

print(pd.DataFrame(features).to_string(index=False))



order_id  current_amount  lag_1_amount  lag_2_amount  lag_3_amount  rolling_mean_3  rolling_max_3
ORD-9341            55.2        210.75         89.00        134.50          144.75         210.75
ORD-9488           178.9         55.20        210.75         89.00          118.32         210.75
ORD-9601           302.1        178.90         55.20        210.75          148.28         210.75


In [5]:
import itertools

orders = [
    {"customer": "C-10482", "category": "electronics", "amount": 349.99},
    {"customer": "C-10482", "category": "electronics", "amount": 189.00},
    {"customer": "C-10482", "category": "apparel",     "amount":  62.50},
    {"customer": "C-10482", "category": "apparel",     "amount":  88.00},
    {"customer": "C-10482", "category": "apparel",     "amount":  45.75},
    {"customer": "C-10482", "category": "home_goods",  "amount": 124.30},
]

# Must be sorted by the grouping key before using groupby
orders_sorted = sorted(orders, key=lambda x: x["category"])

category_features = {}
for category, group in itertools.groupby(orders_sorted, key=lambda x: x["category"]):
    amounts = [o["amount"] for o in group]
    category_features[category] = {
        "order_count":  len(amounts),
        "total_spend":  round(sum(amounts), 2),
        "avg_spend":    round(sum(amounts) / len(amounts), 2),
        "max_spend":    max(amounts),
    }

cat_df = pd.DataFrame(category_features).T
cat_df.index.name = "category"
print(cat_df)


             order_count  total_spend  avg_spend  max_spend
category                                                   
apparel              3.0       196.25      65.42      88.00
electronics          2.0       538.99     269.50     349.99
home_goods           1.0       124.30     124.30     124.30


In [6]:
import itertools

df_poly = pd.DataFrame({
    "avg_order_value":  [142.5, 89.0, 210.3, 67.8],
    "discount_rate":    [0.10,  0.25, 0.05,  0.30],
    "items_per_order":  [3.2,   1.8,  5.1,   1.2],
})

cols = df_poly.columns.tolist()

# Degree-2: includes col^2 and col_a × col_b
for col_a, col_b in itertools.combinations_with_replacement(cols, 2):
    feature_name = f"{col_a}^2" if col_a == col_b else f"{col_a}_x_{col_b}"
    df_poly[feature_name] = df_poly[col_a] * df_poly[col_b]

poly_cols = [c for c in df_poly.columns if "^2" in c or "_x_" in c]
print(df_poly[poly_cols].round(3))



   avg_order_value^2  avg_order_value_x_discount_rate  \
0           20306.25                           14.250   
1            7921.00                           22.250   
2           44226.09                           10.515   
3            4596.84                           20.340   

   avg_order_value_x_items_per_order  discount_rate^2  \
0                             456.00            0.010   
1                             160.20            0.062   
2                            1072.53            0.003   
3                              81.36            0.090   

   discount_rate_x_items_per_order  items_per_order^2  
0                            0.320              10.24  
1                            0.450               3.24  
2                            0.255              26.01  
3                            0.360               1.44  


In [7]:
import itertools
import operator

# Customer C-20917: chronological order amounts
order_amounts = [56.80, 123.40, 89.90, 245.00, 67.50, 310.20, 88.75]

# Cumulative spend
cumulative_spend = list(itertools.accumulate(order_amounts))

# Cumulative max spend (highest single order so far)
cumulative_max = list(itertools.accumulate(order_amounts, func=max))

# Cumulative order count (just using addition on 1s)
cumulative_count = list(itertools.accumulate([1] * len(order_amounts)))

features_df = pd.DataFrame({
    "order_number":    range(1, len(order_amounts) + 1),
    "order_amount":    order_amounts,
    "cumulative_spend": cumulative_spend,
    "cumulative_max_order": cumulative_max,
    "order_count_so_far":   cumulative_count,
})

features_df["avg_spend_so_far"] = (
    features_df["cumulative_spend"] / features_df["order_count_so_far"]
).round(2)

print(features_df.to_string(index=False))


 order_number  order_amount  cumulative_spend  cumulative_max_order  order_count_so_far  avg_spend_so_far
            1         56.80             56.80                  56.8                   1             56.80
            2        123.40            180.20                 123.4                   2             90.10
            3         89.90            270.10                 123.4                   3             90.03
            4        245.00            515.10                 245.0                   4            128.78
            5         67.50            582.60                 245.0                   5            116.52
            6        310.20            892.80                 310.2                   6            148.80
            7         88.75            981.55                 310.2                   7            140.22
